# Palabras de activación propias para Drakter («Jarvis», «Drakter», «despierta»)

Este cuaderno entrena **un modelo de openWakeWord por palabra**, en castellano de España, compatible con el
`openwakeword==0.4.0` y el `onnxruntime` que usa Drakter (Python 3.12). Al final obtienes
`jarvis_es.onnx`, `drakter_es.onnx` y `despierta_es.onnx`.

**Cómo usarlo**

1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU* (T4 o L4 bastan).
2. Revisa la celda **1. Configuración** (palabras, número de muestras, horas de audio).
3. *Entorno de ejecución → Ejecutar todas*. Con los valores por defecto tarda alrededor de 1-1,5 h
   (casi todo es la síntesis de voz, que va en CPU, y las descargas); entrenar cada palabra lleva pocos minutos.
4. Descarga los `.onnx` (se guardan también en tu Drive, en `Drakter/wakeword/`) y sigue la última celda.

**Qué hace**, siguiendo el método del cuaderno oficial `automatic_model_training.ipynb` de openWakeWord, pero en castellano:

| Paso | Oficial (inglés) | Aquí (es-ES) |
|---|---|---|
| Positivos sintéticos | `piper-sample-generator` con LibriTTS en inglés | Piper `es_ES` (6 voces) + Kokoro `es` (3 voces y mezclas), varias velocidades |
| Negativos | ACAV100M precalculado (2000 h, CC BY-NC-SA) | Voz castellana real (MLS español), AudioSet, frases normales y palabras parecidas sintetizadas |
| Aumentación | Ruido de fondo + reverberación (RIR del MIT) | Igual: ruido/voz/música de fondo con SNR aleatoria, RIR del MIT, ecualización y ganancia |
| Características | `melspectrogram.onnx` + `embedding_model.onnx` | **Los mismos ficheros que trae la rueda `openwakeword==0.4.0`** (idénticos a los de Drakter) |
| Modelo | DNN (Flatten → Linear → LayerNorm → ReLU → … → Sigmoid), entrada `(1, 16, 96)` | Igual, exportado a ONNX (opset 13) |
| Métricas | Falsos positivos/hora y recall | Igual, medidos en voz castellana que el modelo no ha visto |

**Licencias de lo que descarga** (para uso personal, no se redistribuye nada):
openWakeWord 0.4.0 — código Apache-2.0, modelos de características Apache-2.0 (el `embedding_model` deriva de
`google/speech_embedding`, Apache-2.0) · piper-tts 1.8.0 — GPL-3.0 (solo se usa como herramienta) ·
voces Piper: carlfm dominio público, davefx CC0, mls_9972/mls_10246 CC BY 4.0, sharvard CC BY 3.0 ·
Kokoro-82M — Apache-2.0 · Multilingual LibriSpeech — CC BY 4.0 · AudioSet (copia `agkphysics/AudioSet`) — CC BY 4.0
(los audios proceden de YouTube) · RIR del MIT (`davidscripka/MIT_environmental_impulse_responses`) — licencia
no indicada en la copia de Hugging Face (McDermott Lab, IR Survey) · opcional ACAV100M precalculado
(`davidscripka/openwakeword_features`) — **CC BY-NC-SA 4.0**, desactivado por defecto.
Detalle en `tools/wakeword/README.md` del repositorio.

## 1. Configuración

Todo lo que puedes tocar está aquí. Los nombres de los modelos (`jarvis_es`, …) son los que luego pondrás en
`DRAKTER_VOICE_WAKE_MODELS`. Las grabaciones reales de tu voz (opcional, ver `tools/wakeword/grabar_muestras.py`)
se buscan en `MUESTRAS_ZIP`.

In [ ]:
import os

IDIOMA = "es-ES"  # castellano de España: voces Piper es_ES y fonemas espeak «es» en Kokoro

PALABRAS = {
    "jarvis_es": {
        # Textos que se sintetizan como positivos. La palabra debe ir AL FINAL (el modelo detecta su final).
        "textos": ["Jarvis", "Jarvis.", "¡Jarvis!", "¿Jarvis?", "Jarvis...", "Yarvis",
                   "Oye, Jarvis", "Hey, Jarvis", "Eh, Jarvis"],
        # Palabras y expresiones parecidas que NO deben activarlo (negativos difíciles).
        "parecidas": ["jardín", "jarabe", "Travis", "Harvey", "Javier", "jabón", "Elvis", "jarra", "ya vi",
                      "barniz", "jarcia", "Marvin", "servís", "Chávez", "jarrón", "Garbí", "Arvizu"],
        "muestras": "jarvis",  # carpeta dentro del ZIP de grabaciones reales
    },
    "drakter_es": {
        "textos": ["Drakter", "Drakter.", "¡Drakter!", "¿Drakter?", "Drákter", "Oye, Drakter", "Eh, Drakter"],
        "parecidas": ["Drácula", "carácter", "tractor", "doctor", "Draco", "Dexter", "dragón", "actor",
                      "práctica", "Baxter", "director", "trajeron", "drástico", "detector", "Dakar"],
        "muestras": "drakter",
    },
    "despierta_es": {
        "textos": ["Despierta", "Despierta.", "¡Despierta!", "Despierta...", "Venga, despierta",
                   "Oye, despierta", "¡Eh, despierta!"],
        "parecidas": ["despiertas", "despierto", "puerta", "desierta", "despeja", "experta", "abierta",
                      "despensa", "cubierta", "despedida", "esperta", "despierte", "la puerta abierta",
                      "despiste", "deprisa"],
        "muestras": "despierta",
    },
}

# Frases normales en castellano: negativos comunes a todos los modelos.
FRASES = [
    "¿Qué tiempo hace hoy en Madrid?", "Pon una alarma a las siete y media.", "Me voy a tomar un café.",
    "Mañana tengo examen de matemáticas.", "¿Has visto mis llaves?", "Vamos al jardín a regar las plantas.",
    "La jarra de agua está en la nevera.", "Travis Scott saca disco nuevo.", "Javier llega tarde otra vez.",
    "Drácula es una novela de Bram Stoker.", "El tractor está aparcado en la entrada.",
    "Cierra la puerta, que hace frío.", "¿Despiertas tú a los niños o lo hago yo?", "Estoy despierto desde las seis.",
    "Ese personaje tiene mucho carácter.", "El doctor me ha dicho que descanse.", "Apunta comprar pan y leche.",
    "¿A qué hora empieza el partido?", "Hoy no me apetece cocinar.", "Ya vi la película que me dijiste.",
    "El jarabe para la tos sabe fatal.", "Necesito terminar el trabajo antes del viernes.",
    "Llama a mi madre cuando puedas.", "La reunión se ha cambiado al martes.", "Qué sueño tengo hoy.",
    "Baja el volumen de la música, por favor.", "Este verano iremos a la playa.", "No encuentro el cargador del móvil.",
    "¿Cuánto cuesta el billete de tren?", "Mi hermano estudia ingeniería en Valencia.",
    "La calle estaba desierta a esas horas.", "Es una experta en historia del arte.", "Abre la ventana un poco.",
    "Voy a pasear al perro.", "¿Me pasas la sal?", "Hay que despejar la mesa antes de cenar.",
    "El director ha convocado una reunión.", "Me encanta el jabón con olor a lavanda.",
    "Elvis Presley era el rey del rock.", "Harvey trabaja en la oficina de al lado.",
    "Esta práctica de laboratorio es muy larga.", "La despensa está casi vacía.", "Despídete de tu abuela.",
    "Vale, de acuerdo, lo hablamos luego.", "Bueno, pues nada, hasta mañana.", "Oye, ¿tú sabes dónde está el mando?",
    "Hey, ¿qué tal todo?", "Eh, mira esto.", "Venga, vámonos ya.", "Uf, qué calor hace.",
]

N_POSITIVOS = 2500        # clips sintéticos por palabra (10 % se reserva para validar)
N_PARECIDAS = 1500        # clips de palabras parecidas por palabra
N_FRASES = 1000           # clips de frases normales (comunes)
AUMENTOS = 3              # versiones aumentadas (ruido + reverberación) por clip sintético
AUMENTOS_REALES = 25      # versiones aumentadas por grabación real tuya
PROP_PIPER = 0.65         # fracción de clips con Piper (~0,1 s/clip); el resto con Kokoro (~0,6 s/clip, en CPU)

HORAS_VOZ_ES = 12             # voz castellana real (MLS, split «train») como negativo
HORAS_VOZ_ES_VALIDACION = 3   # voz real reservada (split «test», otros hablantes) para falsos positivos/hora
AUDIOSET_PARTES = 4           # partes de AudioSet (~690 MB y 500 clips de 10 s ≈ 1,4 h de ruidos, música y voces cada una)
USAR_ACAV = False             # ACAV100M precalculado (17 GB, CC BY-NC-SA 4.0); ver nota en la sección 5

SNR_MIN, SNR_MAX = 0.0, 20.0  # dB de la palabra sobre el ruido de fondo
PASOS = 12000
EVAL_CADA = 500
CAPA = 128                # neuronas por capa (el oficial usa 32 por defecto; 128 da más margen)
BLOQUES = 1
LR = 1e-4
PESO_NEG_MAX = 1000       # rampa del peso de los negativos, como train.py
OBJETIVO_FP_HORA = 0.5    # falsos positivos por hora aceptables
UMBRALES = (0.3, 0.5, 0.7)  # 0.3 es el umbral por defecto de Drakter

MUESTRAS_ZIP = "/content/drive/MyDrive/Drakter/muestras_drakter.zip"  # opcional (grabar_muestras.py)
SUBIR_MUESTRAS = False    # True: te pide el ZIP con un botón en vez de leerlo de Drive
USAR_DRIVE = True         # guarda los resultados en MyDrive/Drakter/wakeword/
SEMILLA = 1234

PIPER_VOCES = ["es_ES-davefx-medium", "es_ES-sharvard-medium", "es_ES-carlfm-x_low",
               "es_ES-mls_9972-low", "es_ES-mls_10246-low"]
KOKORO_VOCES_ES = ["ef_dora", "em_alex", "em_santa"]

# Prueba corta en CPU (la usa la comprobación del repositorio; en Colab no se activa).
PRUEBA_RAPIDA = os.environ.get("DRAKTER_WW_PRUEBA") == "1"
if PRUEBA_RAPIDA:
    N_POSITIVOS, N_PARECIDAS, N_FRASES, AUMENTOS, AUMENTOS_REALES = 30, 16, 16, 1, 2
    HORAS_VOZ_ES, HORAS_VOZ_ES_VALIDACION, AUDIOSET_PARTES = 0.05, 0.03, 0
    PASOS, EVAL_CADA = 300, 100
    USAR_DRIVE = False

## 2. Instalación y entorno

In [ ]:
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # Versiones fijas: openwakeword 0.4.0 es la que usa Drakter (sus melspectrogram/embedding van en la rueda).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openwakeword==0.4.0", "piper-tts==1.8.0",
                    "kokoro-onnx==0.6.1", "soundfile", "onnx", "onnxscript", "pyarrow", "huggingface_hub"],
                   check=True)

import copy, io, json, math, random, shutil, tarfile, time, urllib.request, zipfile
from pathlib import Path, PurePosixPath

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, fftconvolve, resample_poly, sosfilt
from tqdm.auto import tqdm

from openwakeword.utils import AudioFeatures

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
RNG = np.random.default_rng(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from importlib.metadata import version
print("openwakeword", version("openwakeword"), "| torch", torch.__version__, "| dispositivo", DEVICE)
if DEVICE == "cpu" and not PRUEBA_RAPIDA:
    print("⚠️  Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU. En CPU funciona, pero tarda mucho más.")

if EN_COLAB:
    TRABAJO = Path("/content/drakter_ww")
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
else:
    TRABAJO = Path(os.environ.get("DRAKTER_WW_DIR", Path.home() / ".local/share/drakter/wakeword/trabajo"))
TRABAJO.mkdir(parents=True, exist_ok=True)
DESCARGAS = TRABAJO / "descargas"
DESCARGAS.mkdir(exist_ok=True)
SALIDA = TRABAJO / "modelos"
SALIDA.mkdir(exist_ok=True)
NCPU = os.cpu_count() or 2
print("Carpeta de trabajo:", TRABAJO)

## 3. Utilidades de audio (16 kHz, clips de 2 s como openWakeWord)

In [ ]:
SR = 16000
CLIP = 32000   # 2 s → exactamente 16 ventanas de embedding (la entrada del modelo)
FRAME_S = 0.08  # cada ventana de embedding avanza 80 ms


def bajar(url, destino):
    destino = Path(destino)
    if destino.exists() and destino.stat().st_size > 0:
        return destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_name(destino.name + ".part")
    with urllib.request.urlopen(url, timeout=300) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, 1 << 20)
    tmp.rename(destino)
    return destino


def a_16k(x, sr):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(int(sr), SR)
        x = resample_poly(x, SR // g, int(sr) // g).astype(np.float32)
    return x


def leer_audio(datos):
    x, sr = sf.read(io.BytesIO(datos), dtype="float32", always_2d=False)
    return a_16k(x, sr)


def recortar_silencio(x, umbral_db=-40.0):
    t = 160
    n = len(x) // t
    if n == 0:
        return x
    e = np.sqrt((x[: n * t].reshape(n, t) ** 2).mean(1) + 1e-12)
    ok = np.where(20 * np.log10(e / (e.max() + 1e-12)) > umbral_db)[0]
    if not len(ok):
        return x
    return x[max(0, ok[0] - 2) * t: min(n, ok[-1] + 3) * t]


def colocar(palabra, rng):
    # La palabra termina cerca del final del clip (0-0,25 s antes), como en el entrenamiento oficial.
    x = palabra[-(CLIP - 800):]
    fin = CLIP - int(rng.integers(0, int(0.25 * SR)))
    out = np.zeros(CLIP, np.float32)
    ini = max(0, fin - len(x))
    out[ini:fin] = x[len(x) - (fin - ini):]
    return out


def ruidos_sinteticos(rng, n=12, dur=10):
    out = []
    for i in range(n):
        w = rng.standard_normal(SR * dur).astype(np.float32)
        color = i % 3  # blanco, rosa, marrón
        if color:
            spec = np.fft.rfft(w)
            f = np.fft.rfftfreq(len(w), 1 / SR)
            f[0] = 1.0
            spec /= f ** (0.5 * color)
            w = np.fft.irfft(spec, n=len(w)).astype(np.float32)
        out.append(w / (np.abs(w).max() + 1e-9))
    return out


def trozo(f, n, rng):
    if len(f) > n:
        i = int(rng.integers(0, len(f) - n))
        seg = f[i: i + n]
    else:
        seg = np.resize(f, n)
    seg = seg.astype(np.float32)
    return seg / 32768.0 if f.dtype == np.int16 else seg


def aumentar(x, rng, fondos, rirs, con_fondo=0.9):
    # Reverberación, ecualización de micrófono, fondo con SNR aleatoria y ganancia. Devuelve int16.
    y = x.astype(np.float32).copy()
    voz = np.abs(y) > 1e-4
    if rirs and rng.random() < 0.5:
        rir = rirs[int(rng.integers(len(rirs)))]
        r = fftconvolve(y, rir)[: len(y)]
        y = r * (np.sqrt((y ** 2).mean() + 1e-12) / np.sqrt((r ** 2).mean() + 1e-12))
    if rng.random() < 0.3:
        lo, hi = rng.uniform(80, 400), rng.uniform(3000, 7600)
        y = sosfilt(butter(2, [lo, hi], btype="band", fs=SR, output="sos"), y).astype(np.float32)
    if fondos and rng.random() < con_fondo:
        seg = trozo(fondos[int(rng.integers(len(fondos)))], len(y), rng)
        p_voz = (y[voz] ** 2).mean() + 1e-12 if voz.any() else 1e-6
        p_f = (seg ** 2).mean() + 1e-12
        snr = rng.uniform(SNR_MIN, SNR_MAX)
        y = y + seg * np.sqrt(p_voz / (p_f * 10 ** (snr / 10)))
    y = y / (np.abs(y).max() + 1e-9) * rng.uniform(0.05, 0.9)
    return (y * 32767).astype(np.int16)


FEAT = AudioFeatures(ncpu=NCPU)  # melspectrogram.onnx + embedding_model.onnx de la rueda 0.4.0 (los de Drakter)


def _embed(x):
    # Un solo lote por llamada y ncpu <= nº de clips: openwakeword 0.4.0 falla si un lote es menor que ncpu
    return FEAT.embed_clips(x, batch_size=len(x), ncpu=max(1, min(NCPU, len(x))))


def embeber(clips):
    # (N, CLIP) int16 → (N, 16, 96) float16
    clips = np.asarray(clips, dtype=np.int16)
    if not len(clips):
        return np.zeros((0, 16, 96), np.float16)
    out = [_embed(clips[i: i + 512])[:, -16:] for i in range(0, len(clips), 512)]
    return np.concatenate(out).astype(np.float16)


def embeber_largo(audio, seg=30):
    # Audio largo (int16) → (M, T, 96) float16, trozos de `seg` s. Para negativos y falsos positivos/hora.
    n = SR * seg
    k = len(audio) // n
    if k == 0:
        return np.zeros((0, 16, 96), np.float16)
    x = np.asarray(audio[: k * n], dtype=np.int16).reshape(k, n)
    out = [_embed(x[i: i + 64]) for i in range(0, k, 64)]
    return np.concatenate(out).astype(np.float16)


print("Prueba de características:", embeber(np.zeros((1, CLIP), np.int16)).shape)  # (1, 16, 96)

## 4. Descargas: voces, reverberaciones y audio real

Todo lo descargado se trata como dato: los `.tar.gz` y `.parquet` se leen en memoria, sin extraer ni ejecutar nada.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

# Voces Piper es_ES
PIPER = []
from piper import PiperVoice, SynthesisConfig
for nombre in PIPER_VOCES:
    lang, voz, calidad = nombre.split("-")
    base = f"https://huggingface.co/rhasspy/piper-voices/resolve/main/{lang.split('_')[0]}/{lang}/{voz}/{calidad}/{nombre}"
    onnx_p = bajar(base + ".onnx", DESCARGAS / "piper" / f"{nombre}.onnx")
    bajar(base + ".onnx.json", DESCARGAS / "piper" / f"{nombre}.onnx.json")
    v = PiperVoice.load(str(onnx_p))
    for sid in range(max(1, v.config.num_speakers)):
        PIPER.append((nombre, v, sid if v.config.num_speakers > 1 else None))
print("Hablantes Piper:", [(n, s) for n, _, s in PIPER])

# Kokoro (voces es y mezclas con el resto de voces para tener más timbres)
from kokoro_onnx import Kokoro
KREL = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/"
KOKORO = Kokoro(str(bajar(KREL + "kokoro-v1.0.onnx", DESCARGAS / "kokoro" / "kokoro-v1.0.onnx")),
                str(bajar(KREL + "voices-v1.0.bin", DESCARGAS / "kokoro" / "voices-v1.0.bin")))
KOKORO_TODAS = list(KOKORO.get_voices())
print("Voces Kokoro:", len(KOKORO_TODAS), "| es:", [v for v in KOKORO_VOCES_ES if v in KOKORO_TODAS])

# Respuestas al impulso de salas (MIT)
rir_dir = snapshot_download("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset",
                            local_dir=str(DESCARGAS / "mit_rirs"))
RIRS = []
for p in sorted(Path(rir_dir).rglob("*.wav")):
    r = leer_audio(p.read_bytes())
    RIRS.append(r / (np.abs(r).max() + 1e-9))
print("RIR:", len(RIRS))

# Voz castellana real: Multilingual LibriSpeech (CC BY 4.0). Train y test tienen hablantes distintos.
MLS = "facebook/multilingual_librispeech"
_mls_files = [f for f in HfApi().list_repo_files(MLS, repo_type="dataset")
              if f.startswith("data/mls_spanish/") and "/audio/" in f and f.endswith(".tar.gz")]


def voz_mls(split, horas, rng):
    files = [f for f in _mls_files if f"/{split}/" in f]
    rng.shuffle(files)
    trozos, total = [], 0.0
    barra = tqdm(total=horas, desc=f"MLS {split} (h)")
    for f in files:
        if total >= horas:
            break
        try:
            tf = tarfile.open(hf_hub_download(MLS, f, repo_type="dataset", local_dir=str(DESCARGAS / "mls")))
        except tarfile.ReadError:
            continue
        with tf:
            for m in tf.getmembers():
                if total >= horas:
                    break
                if not (m.isfile() and m.name.endswith(".flac")):
                    continue
                x = leer_audio(tf.extractfile(m).read())
                trozos.append((np.clip(x, -1, 1) * 32767).astype(np.int16))
                total += len(x) / SR / 3600
                barra.update(len(x) / SR / 3600)
    barra.close()
    return np.concatenate(trozos) if trozos else np.zeros(0, np.int16)


VOZ_TRAIN = voz_mls("train", HORAS_VOZ_ES, RNG)
VOZ_VAL = voz_mls("test", HORAS_VOZ_ES_VALIDACION, RNG)
print(f"Voz real: {len(VOZ_TRAIN) / SR / 3600:.2f} h entreno, {len(VOZ_VAL) / SR / 3600:.2f} h validación")

# AudioSet (CC BY 4.0): ruidos, música, voces en otros idiomas. Se reserva un 15 % para validar.
AUDIOSET_TRAIN, AUDIOSET_VAL = [], []
if AUDIOSET_PARTES:
    import pyarrow.parquet as pq
    for i in range(AUDIOSET_PARTES):
        p = hf_hub_download("agkphysics/AudioSet", f"data/bal_train/{i:02d}.parquet", repo_type="dataset",
                            local_dir=str(DESCARGAS / "audioset"))
        filas = pq.read_table(p, columns=["audio"]).column("audio").to_pylist()
        for j, a in enumerate(tqdm(filas, desc=f"AudioSet {i:02d}")):
            try:
                x = leer_audio(a["bytes"])
            except Exception:
                continue
            x = (np.clip(x, -1, 1) * 32767).astype(np.int16)
            (AUDIOSET_VAL if j % 7 == 0 else AUDIOSET_TRAIN).append(x)
        del filas
print("AudioSet:", len(AUDIOSET_TRAIN), "clips entreno,", len(AUDIOSET_VAL), "validación")

# Fondos para la aumentación: AudioSet, voz real y ruido sintético
FONDOS = AUDIOSET_TRAIN + [VOZ_TRAIN[i: i + SR * 20] for i in range(0, len(VOZ_TRAIN), SR * 20 * 7)]
FONDOS += ruidos_sinteticos(RNG)
print("Fondos:", len(FONDOS))

## 5. Síntesis de voz en castellano (positivos y negativos)

Varias voces (Piper es_ES con sus dos hablantes de *sharvard*, Kokoro `ef_dora`/`em_alex`/`em_santa` y mezclas
aleatorias de estilos de Kokoro pronunciando en español), velocidades entre 0,75× y 1,35× y variación de entonación.
Fish Audio no se usa: necesita clave y red, y su modelo gratuito tiene fecha de caducidad.

*Nota sobre ACAV100M:* el entrenamiento oficial usa 2000 h de características precalculadas (CC BY-NC-SA 4.0), pero
están calculadas con un `embedding_model.onnx` distinto del que trae `openwakeword==0.4.0` (correlación ≈0,99, no
idéntico), así que por defecto se calculan todos los negativos aquí, con el mismo embedding que usará Drakter.

In [ ]:
def decir(texto, rng):
    # Sintetiza `texto` con una voz y unos parámetros al azar → float32 16 kHz sin silencios.
    if PIPER and (rng.random() < PROP_PIPER or not KOKORO_TODAS):
        _, v, sid = PIPER[int(rng.integers(len(PIPER)))]
        cfg = SynthesisConfig(speaker_id=sid, length_scale=float(rng.uniform(0.75, 1.35)),
                              noise_scale=float(rng.uniform(0.4, 0.9)), noise_w_scale=float(rng.uniform(0.5, 1.0)))
        chunks = list(v.synthesize(texto, syn_config=cfg))
        if not chunks:
            return np.zeros(0, np.float32)
        x, sr = np.concatenate([c.audio_float_array for c in chunks]), chunks[0].sample_rate
    else:
        if rng.random() < 0.5:
            estilo = KOKORO_VOCES_ES[int(rng.integers(len(KOKORO_VOCES_ES)))]
        else:  # mezcla de 2-3 estilos: un timbre nuevo con fonemas españoles
            nombres = list(rng.choice(KOKORO_TODAS, size=int(rng.integers(2, 4)), replace=False))
            nombres[0] = KOKORO_VOCES_ES[int(rng.integers(len(KOKORO_VOCES_ES)))]
            w = rng.dirichlet(np.ones(len(nombres))).astype(np.float32)
            estilo = sum(wi * KOKORO.get_voice_style(n) for wi, n in zip(w, nombres))
        x, sr = KOKORO.create(texto, voice=estilo, speed=float(rng.uniform(0.8, 1.25)), lang="es")
    return recortar_silencio(a_16k(x, sr))


def sintetizar(textos, n, desc, max_s=1.9):
    out = []
    barra = tqdm(total=n, desc=desc)
    intentos = 0
    while len(out) < n and intentos < n * 3:
        intentos += 1
        x = decir(textos[int(RNG.integers(len(textos)))], RNG)
        if 0.15 * SR < len(x) <= max_s * SR:
            out.append(x)
            barra.update(1)
    barra.close()
    return out


t0 = time.time()
TTS = {}
for k, cfg in PALABRAS.items():
    TTS[k] = {"pos": sintetizar(cfg["textos"], N_POSITIVOS, f"{k}: positivos"),
              "parecidas": sintetizar(cfg["parecidas"], N_PARECIDAS, f"{k}: parecidas")}
TTS_FRASES = sintetizar(FRASES, N_FRASES, "frases normales", max_s=6.0)
print(f"Síntesis terminada en {(time.time() - t0) / 60:.1f} min")

# Escucha un par de ejemplos para comprobar la pronunciación
if EN_COLAB:
    from IPython.display import Audio, display
    for k in PALABRAS:
        display(Audio(TTS[k]["pos"][0], rate=SR))

## 6. (Opcional) Tus grabaciones reales

Si grabaste muestras con `tools/wakeword/grabar_muestras.py`, sube `muestras_drakter.zip` a
`MyDrive/Drakter/` (o pon `SUBIR_MUESTRAS = True`). Un 30 % se reserva para medir el *recall* con tu voz; el resto
entra al entrenamiento con aumentación. La carpeta `negativas/` (si la grabaste) se usa como voz que no debe activar.

In [ ]:
REALES = {k: [] for k in PALABRAS}
REALES_NEG = []
zip_datos = None
if EN_COLAB and SUBIR_MUESTRAS:
    from google.colab import files
    subido = files.upload()
    zip_datos = next(iter(subido.values()), None)
elif Path(MUESTRAS_ZIP).is_file():
    zip_datos = Path(MUESTRAS_ZIP).read_bytes()

if zip_datos:
    carpeta_a_modelo = {cfg["muestras"]: k for k, cfg in PALABRAS.items()}
    with zipfile.ZipFile(io.BytesIO(zip_datos)) as z:  # se lee en memoria; solo .wav
        for nombre in z.namelist():
            partes = PurePosixPath(nombre).parts
            if not nombre.lower().endswith(".wav") or len(partes) < 2:
                continue
            x = leer_audio(z.read(nombre))
            if partes[-2] == "negativas":
                REALES_NEG.append((np.clip(x, -1, 1) * 32767).astype(np.int16))
            elif partes[-2] in carpeta_a_modelo:
                x = recortar_silencio(x)
                if 0.15 * SR < len(x) <= 1.9 * SR:
                    REALES[carpeta_a_modelo[partes[-2]]].append(x)
    print("Grabaciones reales:", {k: len(v) for k, v in REALES.items()}, "| negativas:", len(REALES_NEG))
else:
    print("Sin grabaciones reales (opcional). El modelo se entrena solo con voces sintéticas.")

## 7. Aumentación y características (embeddings de openWakeWord)

In [ ]:
def partir(xs, frac=0.1):
    idx = RNG.permutation(len(xs))
    n_val = max(1, int(len(xs) * frac)) if len(xs) > 1 else 0
    return [xs[i] for i in idx[n_val:]], [xs[i] for i in idx[:n_val]]


def clips_aumentados(xs, veces):
    return [aumentar(colocar(x, RNG), RNG, FONDOS, RIRS) for x in xs for _ in range(veces)]


def frases_en_clips(xs):
    # Frases largas: un trozo de 2 s que acaba en un punto al azar de la frase (negativos de habla continua)
    out = []
    for x in xs:
        if len(x) > CLIP:
            fin = int(RNG.integers(CLIP // 2, len(x) + 1))
            x = x[max(0, fin - CLIP): fin]
        out.append(aumentar(colocar(x, RNG), RNG, FONDOS, RIRS))
    return out


DATOS = {}
fr_tr, fr_val = partir(TTS_FRASES)
F_FRASES_TR = embeber(frases_en_clips(fr_tr) + frases_en_clips(fr_tr))
F_FRASES_VAL = embeber(frases_en_clips(fr_val))
for k in tqdm(PALABRAS, desc="características"):
    pos_tr, pos_val = partir(TTS[k]["pos"])
    par_tr, par_val = partir(TTS[k]["parecidas"])
    re_tr, re_val = partir(REALES[k], 0.3)
    DATOS[k] = {
        "pos_tr": embeber(clips_aumentados(pos_tr, AUMENTOS) + clips_aumentados(re_tr, AUMENTOS_REALES)),
        "pos_val": embeber(clips_aumentados(pos_val, 1)),
        "real_val": embeber([(np.clip(colocar(x, RNG), -1, 1) * 32767).astype(np.int16) for x in re_val]
                            + clips_aumentados(re_val, 2)),
        "par_tr": embeber(clips_aumentados(par_tr, AUMENTOS)),
        "par_val": embeber(clips_aumentados(par_val, 1)),
    }
# Las otras palabras de Drakter también son negativos de cada modelo
for k in PALABRAS:
    otras = [DATOS[o]["pos_tr"] for o in PALABRAS if o != k]
    DATOS[k]["neg_tr"] = np.concatenate([DATOS[k]["par_tr"], F_FRASES_TR, *otras])

# Negativos largos (ventanas aleatorias) y validación de falsos positivos por hora
LARGOS_TR = [embeber_largo(VOZ_TRAIN)]
if AUDIOSET_TRAIN:
    LARGOS_TR.append(embeber_largo(np.concatenate(AUDIOSET_TRAIN)))
if REALES_NEG:
    LARGOS_TR.append(embeber_largo(np.concatenate(REALES_NEG), seg=10))
LARGOS_TR = [a for a in LARGOS_TR if len(a)]
VAL_FP = {"voz_es": embeber_largo(VOZ_VAL)}
if AUDIOSET_VAL:
    VAL_FP["audioset"] = embeber_largo(np.concatenate(AUDIOSET_VAL))
HORAS_VAL = {n: len(a) * 30 / 3600 for n, a in VAL_FP.items()}

if USAR_ACAV:
    acav = bajar("https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
                 "openwakeword_features_ACAV100M_2000_hrs_16bit.npy", DESCARGAS / "acav100m.npy")
    LARGOS_TR.append(np.load(acav, mmap_mode="r"))  # (N, 16, 96): ventanas ya hechas

for k, d in DATOS.items():
    print(k, {n: v.shape[0] for n, v in d.items()})
print("Negativos largos:", [a.shape for a in LARGOS_TR], "| validación FP (h):", HORAS_VAL)

## 8. Entrenamiento

Misma red y misma receta que `openwakeword/train.py`: ejemplos difíciles (negativos con puntuación ≥ 0,001 y
positivos < 0,999), peso de los negativos en rampa de 1 a `PESO_NEG_MAX`, *warmup* + coseno. Cada `EVAL_CADA` pasos se
mide recall y falsos positivos por hora, y se queda el mejor punto que cumpla `OBJETIVO_FP_HORA`.

In [ ]:
class Red(nn.Module):
    def __init__(self, capa=CAPA, bloques=BLOQUES):
        super().__init__()
        capas = [nn.Flatten(), nn.Linear(16 * 96, capa), nn.LayerNorm(capa), nn.ReLU()]
        for _ in range(bloques):
            capas += [nn.Linear(capa, capa), nn.LayerNorm(capa), nn.ReLU()]
        capas += [nn.Linear(capa, 1), nn.Sigmoid()]
        self.red = nn.Sequential(*capas)

    def forward(self, x):
        return self.red(x)


def ventanas_largas(n, rng):
    tam = np.array([a.shape[0] * max(1, a.shape[1] - 15) for a in LARGOS_TR], dtype=np.float64)
    cuantos = rng.multinomial(n, tam / tam.sum())
    out = []
    for a, c in zip(LARGOS_TR, cuantos):
        if not c:
            continue
        m = np.sort(rng.integers(0, a.shape[0], size=c))
        if a.shape[1] == 16:
            out.append(np.asarray(a[m], dtype=np.float32))
        else:
            t = rng.integers(0, a.shape[1] - 15, size=c)
            out.append(np.asarray(a[m[:, None], t[:, None] + np.arange(16)], dtype=np.float32))
    return np.concatenate(out)


@torch.no_grad()
def puntuar(modelo, x, lote=8192):
    modelo.eval()
    out = [modelo(torch.as_tensor(np.asarray(x[i: i + lote], dtype=np.float32), device=DEVICE)).squeeze(1).cpu().numpy()
           for i in range(0, len(x), lote)]
    modelo.train()
    return np.concatenate(out) if out else np.zeros(0)


@torch.no_grad()
def puntuar_secuencias(modelo, seqs, lote=16):
    # (M, T, 96) → (M, T-15): puntuación cada 80 ms, como en streaming
    modelo.eval()
    out = []
    for i in range(0, len(seqs), lote):
        s = torch.as_tensor(np.asarray(seqs[i: i + lote], dtype=np.float32), device=DEVICE)
        w = s.unfold(1, 16, 1).permute(0, 1, 3, 2)  # (m, T-15, 16, 96)
        out.append(modelo(w.reshape(-1, 16, 96)).reshape(w.shape[0], w.shape[1]).cpu().numpy())
    modelo.train()
    return np.concatenate(out) if out else np.zeros((0, 0))


def activaciones(p, umbral, refractario=25):
    # Cuenta activaciones (no ventanas): tras una, se ignoran 2 s, como hace la puerta de Drakter al cambiar de modo
    n = 0
    for fila in p:
        idx = np.flatnonzero(fila >= umbral)
        if len(idx):
            n += 1 + int((np.diff(idx) > refractario).sum())
    return n


def evaluar(modelo, d):
    pv, pr, pp, pf = (puntuar(modelo, d["pos_val"]), puntuar(modelo, d["real_val"]),
                      puntuar(modelo, d["par_val"]), puntuar(modelo, F_FRASES_VAL))
    sec = {n: puntuar_secuencias(modelo, a) for n, a in VAL_FP.items()}
    m = {}
    for u in UMBRALES:
        fp = sum(activaciones(s, u) for s in sec.values())
        m[u] = {
            "recall_sintetico": float((pv >= u).mean()) if len(pv) else None,
            "recall_real": float((pr >= u).mean()) if len(pr) else None,
            "fp_hora": fp / max(sum(HORAS_VAL.values()), 1e-9),
            "fp_hora_voz_es": activaciones(sec["voz_es"], u) / max(HORAS_VAL["voz_es"], 1e-9),
            "parecidas_activan": float((pp >= u).mean()) if len(pp) else None,
            "frases_activan": float((pf >= u).mean()) if len(pf) else None,
        }
    return m


def factor_lr(paso):
    calent, mantener = PASOS // 5, PASOS // 3
    if paso < calent:
        return (paso + 1) / calent
    if paso < calent + mantener:
        return 1.0
    return 0.5 * (1 + math.cos(math.pi * (paso - calent - mantener) / max(1, PASOS - calent - mantener)))


def entrenar(k):
    d = DATOS[k]
    pos = torch.as_tensor(np.asarray(d["pos_tr"], dtype=np.float32))
    neg = torch.as_tensor(np.asarray(d["neg_tr"], dtype=np.float32))
    modelo = Red().to(DEVICE)
    opt = torch.optim.Adam(modelo.parameters(), lr=LR)
    pesos = np.linspace(1, PESO_NEG_MAX, PASOS)
    rng = np.random.default_rng(SEMILLA)
    mejor, historial = None, []
    nb_p, nb_n, nb_l = 256, 256, 512
    y = torch.cat([torch.ones(nb_p), torch.zeros(nb_n + nb_l)]).to(DEVICE)
    for paso in tqdm(range(PASOS), desc=f"entrenando {k}"):
        for g in opt.param_groups:
            g["lr"] = LR * factor_lr(paso)
        x = torch.cat([pos[torch.randint(len(pos), (nb_p,))], neg[torch.randint(len(neg), (nb_n,))],
                       torch.as_tensor(ventanas_largas(nb_l, rng))]).to(DEVICE)
        p = modelo(x).squeeze(1)
        dificil = ((y == 0) & (p >= 0.001)) | ((y == 1) & (p < 0.999))
        if dificil.any():
            w = torch.where(y == 1, torch.ones_like(y), torch.full_like(y, float(pesos[paso])))
            perdida = F.binary_cross_entropy(p[dificil], y[dificil], weight=w[dificil])
            opt.zero_grad()
            perdida.backward()
            opt.step()
        if (paso + 1) % EVAL_CADA == 0 or paso + 1 == PASOS:
            m = evaluar(modelo, d)
            r = m[0.5]
            historial.append({"paso": paso + 1, **{str(u): v for u, v in m.items()}})
            cumple = r["fp_hora"] <= OBJETIVO_FP_HORA
            clave = (cumple, r["recall_sintetico"] if cumple else -r["fp_hora"])
            if mejor is None or clave > mejor[0]:
                mejor = (clave, paso + 1, copy.deepcopy(modelo.state_dict()), m)
            print(f"  paso {paso + 1}: recall@0.5={r['recall_sintetico']:.3f} fp/h@0.5={r['fp_hora']:.2f} "
                  f"parecidas@0.5={r['parecidas_activan']:.3f}")
    modelo.load_state_dict(mejor[2])
    return modelo.eval(), mejor[1], mejor[3], historial


MODELOS, METRICAS = {}, {}
for k in PALABRAS:
    MODELOS[k], paso, m, hist = entrenar(k)
    METRICAS[k] = {"paso_elegido": paso, "metricas": {str(u): v for u, v in m.items()}, "historial": hist}

## 9. Métricas: recall y falsos positivos por hora

In [ ]:
def pct(v):
    return "  -  " if v is None else f"{100 * v:5.1f}%"


print(f"Validación de falsos positivos: {HORAS_VAL} h (voz castellana de hablantes no vistos + AudioSet)\n")
recomendado = {}
for k in PALABRAS:
    print(f"== {k} (paso {METRICAS[k]['paso_elegido']})")
    print("  umbral | recall sint. | recall tu voz | FP/h total | FP/h voz es | parecidas | frases")
    for u in UMBRALES:
        r = METRICAS[k]["metricas"][str(u)]
        print(f"  {u:6.2f} |    {pct(r['recall_sintetico'])}    |    {pct(r['recall_real'])}     |"
              f"   {r['fp_hora']:6.2f}   |   {r['fp_hora_voz_es']:6.2f}    |  {pct(r['parecidas_activan'])}  | {pct(r['frases_activan'])}")
    validos = [u for u in UMBRALES if METRICAS[k]["metricas"][str(u)]["fp_hora"] <= OBJETIVO_FP_HORA]
    recomendado[k] = min(validos) if validos else max(UMBRALES)
UMBRAL_RECOMENDADO = max(recomendado.values())  # el umbral de Drakter es común a todos los modelos
print(f"\nUmbral recomendado para Drakter: {UMBRAL_RECOMENDADO} (por modelo: {recomendado})")
print("Recall = % de veces que la palabra activa. FP/h = activaciones falsas por hora de audio normal.")
print("Si el recall con tu voz es bajo, graba muestras reales (sección 6) o baja el umbral; si hay muchos FP, súbelo.")

## 10. Exportación a ONNX y comprobación con openWakeWord 0.4.0

In [ ]:
from openwakeword.model import Model as OWWModel
import onnxruntime as ort


def exportar(modelo, ruta):
    m = copy.deepcopy(modelo).cpu().eval()
    x = torch.zeros(1, 16, 96)
    kw = dict(opset_version=13, input_names=["x"], output_names=["score"])
    try:
        torch.onnx.export(m, (x,), str(ruta), dynamo=False, **kw)
    except TypeError:  # torch antiguo sin el parámetro `dynamo`
        torch.onnx.export(m, (x,), str(ruta), **kw)


def streaming_max(oww, audio_int16):
    oww.reset()
    mejor = 0.0
    for i in range(0, len(audio_int16) - 1279, 1280):  # 80 ms, como la puerta de Drakter
        mejor = max(mejor, max(oww.predict(audio_int16[i: i + 1280]).values()))
    return mejor


rng = np.random.default_rng(7)
for k, modelo in MODELOS.items():
    ruta = SALIDA / f"{k}.onnx"
    exportar(modelo, ruta)
    s = ort.InferenceSession(str(ruta), providers=["CPUExecutionProvider"])
    assert s.get_inputs()[0].shape == [1, 16, 96], s.get_inputs()[0].shape
    x = np.asarray(DATOS[k]["pos_val"][:8], dtype=np.float32)
    onnx_p = np.array([s.run(None, {"x": xi[None]})[0][0][0] for xi in x])
    assert np.allclose(onnx_p, puntuar(modelo, x), atol=1e-4), "ONNX y PyTorch no coinciden"

    # Igual que en Drakter: Model(...) de openwakeword 0.4.0, audio en trozos de 80 ms
    oww = OWWModel(wakeword_model_paths=[str(ruta)])
    silencio = np.zeros(SR, np.int16)
    pos_audio = np.concatenate([silencio, aumentar(colocar(TTS[k]["pos"][0], rng), rng, [], []), silencio])
    neg_audio = VOZ_VAL[: SR * 60] if len(VOZ_VAL) >= SR * 60 else VOZ_VAL
    print(f"{ruta.name}: {ruta.stat().st_size / 1024:.0f} KB | máx. con la palabra {streaming_max(oww, pos_audio):.2f}"
          f" | máx. en 60 s de voz normal {streaming_max(oww, neg_audio):.2f}")

meta = {"palabras": {k: PALABRAS[k]["textos"] for k in PALABRAS}, "idioma": IDIOMA, "umbral_recomendado": UMBRAL_RECOMENDADO,
        "openwakeword": "0.4.0", "metricas": METRICAS, "horas_validacion": HORAS_VAL,
        "config": {"N_POSITIVOS": N_POSITIVOS, "N_PARECIDAS": N_PARECIDAS, "N_FRASES": N_FRASES, "PASOS": PASOS,
                   "CAPA": CAPA, "HORAS_VOZ_ES": HORAS_VOZ_ES, "AUDIOSET_PARTES": AUDIOSET_PARTES, "USAR_ACAV": USAR_ACAV}}
(SALIDA / "metricas.json").write_text(json.dumps(meta, indent=1, ensure_ascii=False))

zip_salida = TRABAJO / "palabras_drakter.zip"
with zipfile.ZipFile(zip_salida, "w") as z:
    for p in sorted(SALIDA.glob("*")):
        z.write(p, p.name)
if EN_COLAB and USAR_DRIVE:
    destino = Path("/content/drive/MyDrive/Drakter/wakeword")
    destino.mkdir(parents=True, exist_ok=True)
    for p in sorted(SALIDA.glob("*")):
        shutil.copy2(p, destino / p.name)
    print("Copiado a Drive:", destino)
if EN_COLAB:
    from google.colab import files
    files.download(str(zip_salida))
print("Listo:", sorted(p.name for p in SALIDA.glob("*")))

## 11. Instalar los modelos en Drakter

En tu ordenador (no en Colab), con el `palabras_drakter.zip` descargado:

```bash
mkdir -p ~/.local/share/drakter/models/openwakeword
unzip -o ~/Descargas/palabras_drakter.zip -d ~/.local/share/drakter/models/openwakeword/
```

Actívalos añadiendo a `~/.config/drakter/.env` (usa el umbral recomendado que imprimió la sección 9):

```bash
DRAKTER_VOICE_WAKE_MODELS=jarvis_es,drakter_es,despierta_es
DRAKTER_VOICE_WAKE_THRESHOLD=0.5
```

Cierra Drakter del todo (bandeja → Salir) y ábrelo de nuevo. Los nombres van sin `.onnx`; puedes dejar solo los que
quieras, o mezclar con el inglés: `hey_jarvis_v0.1,jarvis_es`.

- **Se activa sola a menudo:** sube el umbral (0.6, 0.7…).
- **No te oye a la primera:** bájalo (0.4, 0.3), o graba tus muestras y reentrena.
- **Volver al modelo inglés:** borra esas dos líneas (o pon `DRAKTER_VOICE_WAKE_MODELS=hey_jarvis_v0.1`) y reinicia.

Drakter confirma además en el texto que la frase empiece por la palabra (lista `[wake]` de
`voice/conversation/layer0.es.toml`, que ya incluye «drakter», «despierta» y sus variantes). Más detalle en
`docs/usage/palabras-activacion.md`.

In [ ]:
print("Comandos para tu ordenador:\n")
print("mkdir -p ~/.local/share/drakter/models/openwakeword")
print("unzip -o ~/Descargas/palabras_drakter.zip -d ~/.local/share/drakter/models/openwakeword/")
print("\n# en ~/.config/drakter/.env")
print("DRAKTER_VOICE_WAKE_MODELS=" + ",".join(PALABRAS))
print(f"DRAKTER_VOICE_WAKE_THRESHOLD={UMBRAL_RECOMENDADO}")